# Summarising results

This notebook summarises results into a Pandas dataframe which is then
reformatted into a table suitable for publishing.


In [1]:
import json
from collections import defaultdict
from collections.abc import Mapping
from datetime import datetime, timezone
from itertools import product
from pathlib import Path
from typing import Any, Iterable, Literal, TypeAlias, cast

import pandas as pd
from pandas.io.formats.style import Styler

import qaoa_parameter_setting.utils as utils
from qaoa_parameter_setting.utils.database import ResultsDatabase
from qaoa_parameter_setting.utils.database.summary_table_formatter import (
    convert_evaluation_to_multicolumn_latex,
    formatted_styler_for,
)
from qaoa_parameter_setting.utils.types import (
    Depth,
    EvaluationType,
    GraphKey,
    GraphType,
    MethodConfigJSON,
)

In [2]:
# Make sure to change this when regenerating all tables.
PROBLEM_CLASS: Literal["MC", "MIS"] = "MC"

In [3]:
# TABLE_JSON: str | None = f"summary_tables_{PROBLEM_CLASS}.json"
TABLE_JSON: str | None = None
table = ResultsDatabase(TABLE_JSON, problem_class=PROBLEM_CLASS)

## Setup table with training data and min-max cuts.


In [4]:
if TABLE_JSON is None:

    def ignore_pt_aaa_results(filename: str, results: dict[str, Any]) -> bool:
        """Ignore Parameter Transfer results for the tables."""
        if "PT_" in filename:
            return True
        return False

    # Add training data. Only the "best" results are kept, per graph, trainer config
    # file, and depth.
    for path in [
        "../data/training/random_regular",
        "../data/training/heavy_hex",
        "../data/training/line_to_full",
        "../data/training/erdos_renyi",
    ]:
        table.add_data(path, ignore_file_function=ignore_pt_aaa_results)

In [5]:
if TABLE_JSON is None and table.problem_class == "MC":
    # Add min- and max-cut data. If some graph instances do not have a minmax_cuts
    # entry, an error will be thrown later.
    table.add_minmax_cut_data("../data/minmax_cuts/random_regular")
    table.add_minmax_cut_data("../data/minmax_cuts/heavy_hex")
    table.add_minmax_cut_data("../data/minmax_cuts/line_to_full")
    table.add_minmax_cut_data("../data/minmax_cuts/erdos_renyi")

## Identify missing min- and max-cuts data


In [6]:
if table.problem_class == "MC":
    missing_minmax_cuts = table.missing_minmax_cuts()
    if len(missing_minmax_cuts) == 0:
        print("All minmax_cuts data accounted for.")
    else:
        print(
            "The following graphs are missing min- and max-cuts data. "
            + "Generate them with compute_min_max_for_graph.py"
        )
        for _graph in missing_minmax_cuts:
            print("- {}".format(_graph))
else:
    print(f"Min-/Max-cut data not necessary for {table.problem_class} data.")

All minmax_cuts data accounted for.


## List of methods per evaluation type


In [7]:
table.print_methods_by_evaluation()

       MPS (Aer)         |      MPS (Quimb)      |          PP          |         SV        
--------------------------------------------------------------------------------------------
FA_MPSAer_no_opt.json    | FA_MPS_no_opt.json    | FA_PP_no_opt.json    | FA_SV_no_opt.json 
FA_MPSAer_opt.json       | FA_MPS_opt.json       | FA_PP_opt.json       | FA_SV_opt.json    
F_MPSAer.json            | F_MPS.json            | F_PP.json            | F_SV.json         
I_MPSAer.json            | I_MPS.json            | I_PP.json            | I_SV.json         
LR_MPSAer_angle_opt.json | LR_MPS_angle_opt.json | LR_PP_angle_opt.json | LR_SV_opt.json    
LR_MPSAer_opt.json       | LR_MPS_opt.json       | LR_PP_opt.json       | TQA_SV_no_opt.json
TQA_MPSAer_no_opt.json   | RTS_MPS.json          | RTS_PP.json          | TQA_SV_opt.json   
TQA_MPSAer_opt.json      | TQA_MPS_no_opt.json   | TQA_PP_no_opt.json   | TS_SV.json        
                         | TQA_MPS_opt.json      | TQA_PP_opt.json    

## Save the database if we created a new database


In [8]:
if TABLE_JSON is None:
    TABLE_JSON = f"summary_tables_{table.problem_class}.json"
    table.save(TABLE_JSON, overwrite=True)

## Get raw data table


### Define target instances


In [9]:
# Define target instances if we are filtering.
TARGET_INSTANCES: dict[EvaluationType, set[GraphKey] | dict[bool, set[GraphKey]]] | None
"""The target instances for all Summary Tables of this problem class.

We only filter instances here for MIS tables. MAXCUT filtering is done during
table generation using `only_common_instances`.
"""
if table.problem_class == "MIS":
    LARGE_INSTANCES: list[GraphKey] = [
        _graph_format.format(idx)
        for idx in range(10)
        for _graph_format in [
            "{:03}_100nodes_random3regular.json",
            "{:03}_7_3_heavyhex_144nodes_weighted.json",
            "{:03}_100nodes_2swap_layers.json",
            "{:03}_70nodes_erdosrenyi20percent.json",
        ]
    ]
    TARGET_INSTANCES = {
        "MPS": {
            False: set(LARGE_INSTANCES),
            True: set(LARGE_INSTANCES),
        },
        "PP": set(LARGE_INSTANCES),
        "SV": set(
            [
                _graph_format.format(idx)
                for idx in range(10)
                for _graph_format in [
                    "{:03}_20nodes_random3regular.json",
                    "{:03}_1_2_heavyhex_21nodes_weighted.json",
                    "{:03}_20nodes_2swap_layers.json",
                    "{:03}_20nodes_erdosrenyi20percent.json",
                ]
            ]
        ),
    }
else:
    TARGET_INSTANCES = None

### Define reference methods for filtering runs


In [10]:
# Determine reference methods for depth 10 tables.
REFERENCE_METHODS_10: (
    dict[
        EvaluationType | tuple[Literal["MPS"], bool],
        dict[
            GraphType,
            MethodConfigJSON,
        ],
    ]
    | None
)
"""Reference methods for depth 10 tables, only used for MAXCUT data.

These were found by hand, identifying methods with enough runs but that also
reduced the number of additional simulations to run.
"""
if table.problem_class == "MC":
    REFERENCE_METHODS_10 = {
        ("MPS", False): {
            "erdos_renyi": "F_MPS.json",
            "heavy_hex": "F_MPS.json",
            "line_to_full": "LR_MPS_opt.json",
            "random_regular": "FA_MPS_opt.json",
        },
        ("MPS", True): {
            "erdos_renyi": "F_MPSAer.json",
            "heavy_hex": "F_MPSAer.json",
            "line_to_full": "LR_MPSAer_opt.json",
            "random_regular": "FA_MPSAer_opt.json",
        },
        "SV": {
            "erdos_renyi": "TQA_SV_opt.json",
            "heavy_hex": "F_SV.json",
            "line_to_full": "LR_SV_opt.json",
            "random_regular": "TQA_SV_opt.json",
        },
        "PP": {
            "erdos_renyi": "TQA_PP_opt.json",
            "heavy_hex": "TQA_PP_opt.json",
            "line_to_full": "TQA_PP_opt.json",
            "random_regular": "TQA_PP_opt.json",
        },
    }  # type: ignore[assignment]
elif table.problem_class == "MIS":
    REFERENCE_METHODS_10 = None
else:
    raise NotImplementedError(
        "Cannot determine reference methods for problem class {!r}.".format(
            table.problem_class
        )
    )

### Define number of nodes for filtering runs


In [11]:
NUM_NODES: Iterable[int] | dict[str, Iterable[int] | dict[bool, Iterable[int]]] | None
if table.problem_class == "MIS":
    NUM_NODES = {
        "MPS": {True: [70, 100, 144], False: [70, 100, 144]},
        "PP": [70, 100, 144],
        "SV": [20, 21],
    }
else:
    NUM_NODES = None

In [12]:
# Only select results with the best energy per config and instance:
table = table.only_best_parameters("config")

if TARGET_INSTANCES is not None or NUM_NODES is not None:
    table = table.filter_by(
        instance_filter=TARGET_INSTANCES,
        num_nodes=NUM_NODES,
    )


# This is the _raw_ table with all results
df: pd.DataFrame = table.to_dataframe()
df

,instance,num_nodes,graph_type,trainer_config,method,depth,energy,trainer,evaluation,evaluation_label,method_label,with_aer,source_file,train_duration,metadata,result_index,run_datetime,result_key_index,approximation_ratio
0,000_100nodes_random3regular.json,100,random_regular,TQA_PP_opt.json,TQA_opt.json,10,49.411109,ScipyTrainer,PP,PP,TQA*,False,../data/training/random_regular/20250827_17160...,152.254454,"{'iteration': '1', 'version': 13}",0,2025-08-27 17:16:02,1.0,0.908110
1,000_100nodes_random3regular.json,100,random_regular,TQA_PP_no_opt.json,TQA_no_opt.json,10,47.336897,TQATrainer,PP,PP,TQA,False,../data/training/random_regular/20250827_17160...,86.293813,"{'iteration': '0', 'version': 13}",0,2025-08-27 17:16:02,0.0,0.892970
2,000_100nodes_random3regular.json,100,random_regular,FA_PP_opt.json,FA_opt.json,10,51.490788,ScipyTrainer,PP,PP,Fixed Angle*,False,../data/training/random_regular/20250827_18372...,92.740585,"{'iteration': '1', 'version': 13}",0,2025-08-27 18:37:29,1.0,0.923290
3,000_100nodes_random3regular.json,100,random_regular,FA_PP_no_opt.json,FA_no_opt.json,10,50.800136,FixedAngleConjecture,PP,PP,Fixed Angle†,False,../data/training/random_regular/20250827_18372...,4.178287,"{'iteration': '0', 'version': 13}",0,2025-08-27 18:37:29,0.0,0.918249
4,000_100nodes_random3regular.json,100,random_regular,I_PP.json,I.json,1,28.701487,ScipyTrainer,PP,PP,Interp.*,False,../data/training/random_regular/20250827_18494...,0.787144,"{'iteration': '1', 'version': 13}",0,2025-08-27 18:49:42,1.0,0.756945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85331,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,LR_SV_opt.json,LR_opt.json,6,10.761517,TQATrainer,SV,SV,Linear Ramp,False,../data/training/erdos_renyi/20260405_122749_0...,34.562427,"{'iteration': '0', 'version': 33}",0,2026-04-05 12:27:49,0.0,0.927935
85332,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,LR_SV_opt.json,LR_opt.json,7,11.067069,TQATrainer,SV,SV,Linear Ramp,False,../data/training/erdos_renyi/20260405_123346_0...,71.562197,"{'iteration': '0', 'version': 33}",0,2026-04-05 12:33:46,0.0,0.935976
85333,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,LR_SV_opt.json,LR_opt.json,8,11.310784,TQATrainer,SV,SV,Linear Ramp,False,../data/training/erdos_renyi/20260405_123948_0...,39.076143,"{'iteration': '0', 'version': 33}",0,2026-04-05 12:39:48,0.0,0.942389
85334,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,LR_SV_opt.json,LR_opt.json,9,11.464066,TQATrainer,SV,SV,Linear Ramp,False,../data/training/erdos_renyi/20260405_124649_0...,61.933105,"{'iteration': '0', 'version': 33}",0,2026-04-05 12:46:49,0.0,0.946423


## Get formatted and pivoted table and styler

With Pandas, dataframes are formatted with Stylers.
`qaoa_parameter_setting.utils.database.summary_table_formatter.formatted_styler_for` automatically
stylises the Styler and returns the pivoted dataframe and Styler.


### Example summary table with `"text"` formatting

The following cells format the summary results into a single table suitable for
Jupyter notebooks. Later cells will format the table for LaTeX and save them to
a file.

As we can generate tables for both MAXCUT and MIS, we use `performance_metric`
to store the field name used to quantify performance.


In [13]:
performance_metric: Literal["approximation_ratio", "energy"]
if table.problem_class == "MC":
    performance_metric = "approximation_ratio"
elif table.problem_class == "MIS":
    performance_metric = "energy"
else:
    raise NotImplementedError(
        "Performance summary tables not implemented for {!r} problem class.".format(
            table.problem_class
        )
    )

#### MAXCUT Approximation Ratio or MIS Normalized Energy


In [14]:
pivot, styler, _ = formatted_styler_for(
    table,
    depths=10,
    agg_values=performance_metric,
    with_fancy_values=True,
    cmap={"MPS (Aer)": "BuPu", "MPS (Quimb)": "YlOrBr", "PP": "YlGn", "SV": "PuBu"},
    precision=2,
    missing_data_str="-",
    target_format="text",
    show_empty_rows=True,
    # exclude_methods=["RTS", "TS"],
)
styler

#### Number of instances and nodes


In [15]:
pivot, styler, _ = formatted_styler_for(
    table,
    depths=10,
    num_nodes=None,
    agg_values="num_instances",
    with_fancy_values=True,
    cmap="Greens",
    precision=0,
    missing_data_str="-",
    target_format="text",
    show_empty_rows=True,
    # exclude_methods=["RTS", "TS"],
)
styler

## Combined table


In [16]:
table_cmap = {
    (k1, k2): v if k2 == performance_metric else "RdPu"
    for k1, v in {
        "MPS (Aer)": "BuPu",
        "MPS (Quimb)": "YlOrBr",
        "PP": "YlGn",
        "SV": "PuBu",
    }.items()
    for k2 in [performance_metric, "num_instances"]
}
pivot, styler, cmap_ranges = formatted_styler_for(
    table,
    depths=10,
    agg_values=[performance_metric, "num_instances"],
    with_fancy_values=True,
    cmap=table_cmap,
    precision={performance_metric: 2, "num_instances": 0},
    missing_data_str="-",
    target_format="text",
    show_empty_rows=False,
    # exclude_methods=["RTS"],
)
styler

In [17]:
table_cmap = {
    (k1, k2): v if k2 == performance_metric else "RdPu"
    for k1, v in {
        "MPS (Aer)": "BuPu",
        "MPS (Quimb)": "YlOrBr",
        "PP": "YlGn",
        "SV": "PuBu",
    }.items()
    for k2 in [performance_metric, "num_instances"]
}
pivot, styler, cmap_ranges = formatted_styler_for(
    table,
    depths=6,
    num_nodes={
        "MPS (Aer)": [70, 100],
        "MPS (Quimb)": [70, 100],
        "PP": [70, 100],
        "SV": 20,
    },
    agg_values=[performance_metric, "num_instances"],
    with_fancy_values=True,
    cmap=table_cmap,
    precision={performance_metric: 2, "num_instances": 0},
    missing_data_str="-",
    target_format="text",
    # exclude_methods=["RTS"],
)
styler

## Create and save tables for LaTeX

Here we format tables into LaTeX files and save them into separate `.tex` files
suitable for inclusion in a paper. Files are saved with the current date and
time for tracking changes. These can be compiled into a preview of all tables
with `pdflatex summary_tables.tex`


In [18]:
AggValue: TypeAlias = Literal["num_instances", "approximation_ratio", "energy"]
values_to_plot: list[
    tuple[AggValue | Literal["both"], AggValue | list[AggValue], str, str]
] = [
    (
        "num_instances",
        "num_instances",
        "Number of Instances (num. nodes in brackets)",
        " Cells are colored based on the number of instances for the given configuration, with darker colors indicating more instances.",
    )
]
# Define values and captions for tables
if table.problem_class == "MC":
    values_to_plot.extend(
        [
            (
                "approximation_ratio",
                "approximation_ratio",
                r"Avg. Approx. Ratio $\pm$ standard deviation in percentage for MAXCUT",
                " Evaluation methods are shown in different colors, with darker colors indicating better approximation ratios.",
            ),
            (
                "both",
                ["approximation_ratio", "num_instances"],
                "Avg. Approx. Ratio and Number of Instances for Various Evaluation Methods for MAXCUT",
                " Evaluation methods are shown in different colors for approximation ratios."
                + " Darker colors indicating better approximation ratios or more instances."
                + " Approximation ratios are in percentage, with $\\pm$ their standard deviation."
                + " The range of graph sizes are shown in brackets, next to the number of instances.",
            ),
        ]
    )
elif table.problem_class == "MIS":
    values_to_plot.extend(
        [
            (
                "energy",
                "energy",
                r"Avg. Energy $\pm$ standard deviation for MIS",
                " Evaluation methods are shown in different colors, with darker colors indicating better energies."
                + " The minimum and maximum energies defining the range of colors are shared between MPS and PP."
                + " Note that energies include invalid solutions, which are penalized by the Hamiltonian.",
            ),
            (
                "both",
                ["energy", "num_instances"],
                "Avg. Energy and Number of Instances for Various Evaluation Methods for MIS",
                " Evaluation methods are shown in different colors for energies."
                + " Darker colors indicating better energies or more instances."
                + " The minimum and maximum energies defining the range of colors are shared between MPS and PP."
                + " Average energies are shown, with $\\pm$ their standard deviation."
                + " The range of graph sizes are shown in brackets, next to the number of instances.",
            ),
        ]
    )

In [19]:
## Create the CMAPs for each table.
TABLE_CMAPS_PER_EVALUATION = {
    "MPS (Aer)": "BuPu",
    "MPS (Quimb)": "YlOrBr",
    "PP": "YlGn",
    "SV": "PuBu",
}
table_cmap_num_instances = "Greens"
# The combined tables use different ranges for SV. MAXCUT has the same range for all evaluation
# methods, whereas SV has its own range for MIS data.
if table.problem_class == "MC":
    table_cmap_performance = TABLE_CMAPS_PER_EVALUATION
    table_cmap_combined = [
        {
            (str(k1), performance_metric): v
            for k1, v in TABLE_CMAPS_PER_EVALUATION.items()
        },
        {
            (str(k1), "num_instances"): "RdPu"
            for k1 in TABLE_CMAPS_PER_EVALUATION.keys()
        },
    ]
else:
    table_cmap_performance = [
        {"MPS (Aer)": "BuPu", "MPS (Quimb)": "YlOrBr", "PP": "YlGn"},
        {"SV": "PuBu"},
    ]
    table_cmap_combined = [
        {
            (str(k1), performance_metric): v
            for k1, v in TABLE_CMAPS_PER_EVALUATION.items()
            if k1 != "SV"
        },
        {
            (str(k1), performance_metric): v
            for k1, v in TABLE_CMAPS_PER_EVALUATION.items()
            if k1 == "SV"
        },
        {
            (str(k1), "num_instances"): "RdPu"
            for k1 in TABLE_CMAPS_PER_EVALUATION.keys()
        },
    ]

In [20]:
now = datetime.now(tz=timezone.utc)
generated_on_str = "% Generated on {now:%Y-%m-%d} at {now:%H:%M:%S} UTC\n".format(
    now=now
)
# Open a summary list_of_tables file for previewing all tables.
with open("list_of_tables_{}.tex".format(table.problem_class), "w") as f:
    # Write date and time to list_of_tables
    f.write(generated_on_str)

    # Iterate over all depths, sorted so they're included in increasing order in
    # list_of_tables
    for depth in sorted(int(x) for x in table.to_dataframe()["depth"].unique()):
        # Write a section title.
        _ = f.write("\n\\section{{Tables for depth $P={}$}}\n".format(depth))
        for filename_suffix, values, value_label, additional_caption in values_to_plot:
            # This is the filename for this table.
            _latex_filename = "table_{problem_class}_p{depth:02}_{suffix}.tex".format(
                problem_class=table.problem_class,
                depth=int(depth),
                suffix=filename_suffix,
            )
            if REFERENCE_METHODS_10 is not None and depth == 10:
                sub_table = table.only_common_instances(
                    REFERENCE_METHODS_10, per_depth=True
                )
            else:
                sub_table = table
            # Get the styler
            _, styler, _ = formatted_styler_for(
                sub_table,
                depths=depth,
                # Select the number of nodes for MIS, only.
                agg_values=values,
                with_fancy_values=True,
                cmap=(  # pyright: ignore[reportArgumentType]
                    table_cmap_num_instances
                    if values == "num_instances"
                    else table_cmap_performance
                )
                if isinstance(values, str)
                else table_cmap_combined,
                precision={performance_metric: 1, "num_instances": 0},
                missing_data_str="-",
                target_format="latex",
                show_empty_rows=True,
            )

            # Save table to separate LaTeX file
            with open(_latex_filename, "w") as f_table:
                _ = f_table.write(generated_on_str)
                _ = f_table.write(
                    "% Data is {value_label} for depth P={depth}.\n".format(
                        value_label=value_label, depth=depth
                    )
                )
                f_table.write(
                    convert_evaluation_to_multicolumn_latex(
                        styler.to_latex(
                            # f_table,
                            convert_css=True,
                            hrules=True,
                            clines="skip-last;data",
                        ),
                        num_metrics=len(values) if isinstance(values, list) else 1,
                    )
                )
            _ = f.write(
                r"""
\begin{{table}}[H]
    \centering
    \input{{{filename}}}
    \caption{{\textbf{{{value_label} for $P={depth}$.}} Graph types are Erdos Renyi (ER), Heavy-Hex (HH), Line-to-Full (LB), and Random Regular (RR).{additional_caption}}}
\end{{table}}
""".format(
                    filename=_latex_filename,
                    depth=depth,
                    value_label=value_label,
                    additional_caption=additional_caption,
                )
            )
            _ = f.write(r"\clearpage")
        # _ = f.write(r"\clearpage")

## Compiling and previewing all tables

Tables are written to `table_p<depth>_<metric>.tex` where `<depth>` is the QAOA
depth and `<metric>` is either `approximation_ratio` or `num_instances`. These
LaTeX files contain the `tabular` environments to include the given tables.
Numerical values are formatted with siunitx for easier precision and uncertainty
handling. All of these tables are then included in `tables.tex` as a list of
sections, one per depth. `summary_tables.tex` is an example LaTeX document that
(i) has an appropriate preamble for rendering the tables and (ii) shows how to
include them in a larger document.

To compile the _sample document_ `summary_tables.tex`, run the following command
after generating the LaTeX files:

```bash
latexmk -pdf summary_tables.tex
```


In [21]:
!latexmk -pdf summary_tables.tex

Rc files read (in order):
  NONE
Latexmk: This is Latexmk, John Collins, 9 March 2026. Version 4.88.
Latexmk: applying rule 'pdflatex'...
Rule 'pdflatex':  Reasons for rerun
Changed files or newly in use/created:
  list_of_tables_MC.tex
  table_MC_p01_approximation_ratio.tex
  table_MC_p01_both.tex
  table_MC_p01_num_instances.tex
  table_MC_p02_approximation_ratio.tex
  table_MC_p02_both.tex
  table_MC_p02_num_instances.tex
  table_MC_p03_approximation_ratio.tex
  table_MC_p03_both.tex
  table_MC_p03_num_instances.tex
  table_MC_p04_approximation_ratio.tex
  table_MC_p04_both.tex
  table_MC_p04_num_instances.tex
  table_MC_p05_approximation_ratio.tex
  table_MC_p05_both.tex
  table_MC_p05_num_instances.tex
  table_MC_p06_approximation_ratio.tex
  table_MC_p06_both.tex
  table_MC_p06_num_instances.tex
  table_MC_p07_approximation_ratio.tex
  table_MC_p07_both.tex
  table_MC_p07_num_instances.tex
  table_MC_p08_approximation_ratio.tex
  table_MC_p08_both.tex
  table_MC_p08_num_instances

# Missing Files


## Setup


In [22]:
from IPython.display import display

In [23]:
if table.problem_class == "MC":
    missing_depth = 10
elif table.problem_class == "MIS":
    missing_depth = 6
else:
    raise ValueError(
        f"Problem class {table.problem_class!r} not supported for missing-config generation."
    )


## Identify all methods we want to include in the table


In [24]:
# These are the methods we would like to include in our tables. Some are
# 'virtual' _no_opt methods as no method JSON file exists, but we extract the
# data from an intermediate run of an _opt-method run.
target_methods: dict[
    EvaluationType, list[MethodConfigJSON] | dict[bool, list[MethodConfigJSON]]
] = {
    "MPS": {
        True: cast(
            list[MethodConfigJSON],
            [
                "FA_MPSAer_no_opt.json",
                "FA_MPSAer_opt.json",
                "F_MPSAer.json",
                "I_MPSAer.json",
                "LR_MPSAer_angle_opt.json",
                "LR_MPSAer_opt.json",
                "RTS_MPSAer.json",
                "TQA_MPSAer_no_opt.json",
                "TQA_MPSAer_opt.json",
            ],
        ),
        False: cast(
            list[MethodConfigJSON],
            [
                "FA_MPS_no_opt.json",
                "FA_MPS_opt.json",
                "F_MPS.json",
                "I_MPS.json",
                "LR_MPS_angle_opt.json",
                "LR_MPS_opt.json",
                "RTS_MPS.json",
                "TQA_MPS_no_opt.json",
                "TQA_MPS_opt.json",
            ],
        ),
    },
    "PP": cast(
        list[MethodConfigJSON],
        [
            "FA_PP_no_opt.json",
            "FA_PP_opt.json",
            "F_PP.json",
            "I_PP.json",
            "LR_PP_angle_opt.json",
            "LR_PP_opt.json",
            # We don't want Parameter Transfer at all.
            # "PT_PP_AAAM.json",
            "RTS_PP.json",
            "TQA_PP_no_opt.json",
            "TQA_PP_opt.json",
        ],
    ),
    "SV": cast(
        list[MethodConfigJSON],
        [
            "FA_SV_no_opt.json",
            "FA_SV_opt.json",
            "F_SV.json",
            "I_SV.json",
            "LR_SV_angle_opt.json",
            "LR_SV_opt.json",
            "TQA_SV_no_opt.json",
            "TQA_SV_opt.json",
            "TS_SV.json",
        ],
    ),
}

all_methods: list[MethodConfigJSON] = list(
    set(
        method
        for methods in target_methods.values()
        for method in (
            methods
            if isinstance(methods, list)
            else [m for sublist in methods.values() for m in sublist]
        )
    )
)

### Create Row Index for Pivot Tables


In [25]:
# Create multi-index of target evaluation and method labels.
row_index = pd.MultiIndex.from_tuples(
    [
        (
            evaluation_label,
            method_label,
        )
        for evaluation_label, method_label in product(
            ["MPS (Aer)", "MPS (Quimb)", "PP", "SV"],
            sorted(
                set(
                    [
                        utils.labels.trainer_config_to_method_label(m)
                        for m in all_methods
                    ]
                )
            ),
        )
    ],
)

## Filter table for reference methods (MAXCUT only)


In [26]:
# Filter for common instances for MAXCUT P=10 table
if table.problem_class == "MC":
    assert REFERENCE_METHODS_10 is not None
    # We filter by reference methods as table was not filtered by common
    # instances. Only sub_table during LaTeX table generation is filtered, and
    # only for P=10.
    missing_table = table.only_common_instances(
        REFERENCE_METHODS_10,
        per_depth=True,
    )
elif table.problem_class == "MIS":
    # We don't filter by num_nodes as we already did that in the table creation.
    missing_table = table
else:
    raise ValueError(f"Unrecognised problem class {table.problem_class!r}.")

## Pivot Table of Existing Runs


In [27]:
# 1. Show the number of runs
pivot, _, _ = formatted_styler_for(
    missing_table,
    agg_values="num_instances",
    depths=missing_depth,
    show_empty_rows=True,
)
# We need to reprocess the values to floats as formatted_styler_for converts
# them to strings.
pivot = pivot.map(lambda x: float(x))
pivot = pivot.reindex(index=row_index)
styler: Styler = pivot.style.format(na_rep="-", precision=0)  # pyright: ignore[reportAssignmentType]
styler = styler.background_gradient("Greens", axis=None).highlight_null("red")
display(styler)


## Get Missing Instances


### Create Mapping of Target Instances


In [28]:
# Helper functions for saving and loading the target instances to a JSON file.
def __sort_graph_key(graph_key: GraphKey) -> str:
    # The first 4 characters of graph_key are the graph indexes. We move this to
    # the end of the sorting key so we always group (1) the same sized graphs
    # together and (2) graphs of the same type together.
    if len(graph_key) < 4:
        # Catch-all in-case we don't have enough characters. This should never
        # be the case, but we do it anyway.
        return graph_key
    return graph_key[3:] + graph_key[:]


def save_target_instances_json(
    obj: Mapping[EvaluationType, set[GraphKey] | Mapping[bool, set[GraphKey]]], path
):
    output = {}
    for key1, val1 in obj.items():
        if isinstance(val1, set):
            output[key1] = list(sorted(val1, key=__sort_graph_key))
        else:
            output[key1] = {}
            for key2, val2 in val1.items():
                if isinstance(val2, set):
                    output[key1][key2] = list(sorted(val2, key=__sort_graph_key))
                else:
                    output[key1][key2] = val2
    with open(path, "w") as f:
        json.dump(output, f, indent=2)


def load_target_instances_json(
    path,
) -> dict[EvaluationType, set[GraphKey] | dict[bool, set[GraphKey]]]:
    with open(path, "r") as f:
        data = json.load(f)
    output: dict[EvaluationType, set[GraphKey] | dict[bool, set[GraphKey]]] = {}
    key1: EvaluationType
    for key1, val1 in data.items():
        assert isinstance(key1, str)
        if isinstance(val1, list):
            output[key1] = set(val1)
        else:
            output[key1] = {}
            for key2, val2 in val1.items():
                _bool_key2 = key2 == "true"
                if isinstance(val2, list):
                    output[key1][_bool_key2] = set(val2)  # pyright: ignore[reportIndexIssue]
                else:
                    output[key1][_bool_key2] = val2  # pyright: ignore[reportIndexIssue]
    return output


In [29]:
target_instances_for_missing: dict[
    EvaluationType, set[GraphKey] | dict[bool, set[GraphKey]]
] = {}
if missing_table.problem_class == "MC":
    # ========================================
    # Determine the target instances for MAXCUT
    # ========================================
    #
    # This is done in two possible ways:
    # 1. Using `{problem_class}_target_instances.json`.
    # 2. By determining all instances we have in the data and populating the
    #    table with them.
    #
    # Depending on what you are doing, you will choose one of these. Note that
    # we do not use these methods for MIS. Instead we programmatically define
    # those earlier in the notebook.
    #
    # BY DEFAULT YOU SHOULD USE OPTION 1. THIS ENSURES YOU ARE CHECKING THE
    # CORRECT INSTANCES.

    # OPTION 1
    # ========
    target_instances_for_missing = load_target_instances_json(
        f"{table.problem_class}_target_instances.json"
    )

    # OPTION 2
    # ========
    # target_instances_for_missing = {}
    # for _inst, _inst_data in missing_table.data.items():
    #     for trainer_config in _inst_data.keys():
    #         evaluation = utils.labels.trainer_config_to_evaluation(trainer_config)
    #         with_aer = utils.labels.method_uses_aer(trainer_config)
    #         if evaluation not in target_instances_for_missing:
    #             if evaluation == "MPS":
    #                 target_instances_for_missing[evaluation] = {
    #                     True: set(),
    #                     False: set(),
    #                 }
    #             else:
    #                 target_instances_for_missing[evaluation] = set()
    #         if evaluation == "MPS":
    #             target_instances_for_missing[evaluation][with_aer].add(_inst)  # pyright: ignore[reportIndexIssue]
    #         else:
    #             target_instances_for_missing[evaluation].add(_inst)  # pyright: ignore[reportAttributeAccessIssue]

elif missing_table.problem_class == "MIS":
    assert TARGET_INSTANCES is not None
    target_instances_for_missing = TARGET_INSTANCES
else:
    raise ValueError(
        f"Cannot compute target instances for problem class {missing_table.problem_class!r}."
    )

### Load Failed Runs


In [30]:
failed_runs_filename = Path(f"failed_runs_{table.problem_class}.json")
failed_runs: dict[GraphKey, dict[MethodConfigJSON, dict[Depth, str]]]
if failed_runs_filename.exists():
    failed_runs = missing_table.load_failed_configs_from_json(failed_runs_filename)
else:
    failed_runs = defaultdict(lambda: defaultdict(dict))


### Compute Missing Instances Dataframe


In [31]:
missing: dict[
    EvaluationType | tuple[Literal["MPS"], bool],
    dict[MethodConfigJSON, dict[Depth, set[GraphKey]]],
] = missing_table.get_missing_configs(
    target_instances=target_instances_for_missing,
    target_depths=[missing_depth],
    target_methods=target_methods,
    failed_configs=failed_runs,
    # We do not return derived configs as we can extract them from `missing`.
    with_derived_configs=False,
)

missing_records = []
for _eval_type, _eval_data in missing.items():
    if _eval_type in ["SV", "PP"]:
        _evaluation = _eval_type
        _with_aer = None
    elif _eval_type[0] == "MPS":
        _evaluation = _eval_type[0]
        _with_aer = _eval_type[1]
    else:
        raise ValueError(f"Evaluation {_eval_type!r} not recognized.")
    for _trainer_config, _method_data in _eval_data.items():
        for _depth, _instances in _method_data.items():
            assert _depth == missing_depth
            missing_records.extend(
                [
                    {
                        "instance": _inst,
                        "method": _trainer_config,
                        "depth": _depth,
                        "evaluation": _evaluation,
                        "with_aer": _with_aer,
                        "method_label": utils.labels.trainer_config_to_method_label(
                            _trainer_config
                        ),
                        "evaluation_label": utils.labels.trainer_config_to_evaluation_label(
                            _trainer_config
                        ),
                        "graph_type": utils.instance.graph_type(_inst),
                    }
                    for _inst in _instances
                ]
            )
df_missing = pd.DataFrame.from_records(missing_records)
df_missing


/home/conrad/Projects/QAOA_Parameter_Setting/QAOA-Parameter-Setting/qaoa_parameter_setting/utils/database/results_database.py:1968: UserWarning: Method 'RTS_MPSAer.json' for evaluation 'MPS' (with Aer) has no data in the database
  warnings.warn(
/home/conrad/Projects/QAOA_Parameter_Setting/QAOA-Parameter-Setting/qaoa_parameter_setting/utils/database/results_database.py:1972: UserWarning: Method 'LR_SV_angle_opt.json' for evaluation 'SV' has no data in the database
  warnings.warn(


,instance,method,depth,evaluation,with_aer,method_label,evaluation_label,graph_type
0,000_4_1_heavyhex_39nodes_weighted.json,RTS_MPS.json,10,MPS,False,Recursive TS,MPS (Quimb),heavy_hex
1,009_4_1_heavyhex_39nodes_weighted.json,RTS_MPS.json,10,MPS,False,Recursive TS,MPS (Quimb),heavy_hex
2,008_7_2_heavyhex_105nodes_weighted.json,RTS_MPS.json,10,MPS,False,Recursive TS,MPS (Quimb),heavy_hex
3,001_100nodes_random8regular.json,RTS_MPS.json,10,MPS,False,Recursive TS,MPS (Quimb),random_regular
4,008_40nodes_erdosrenyi20percent.json,RTS_MPS.json,10,MPS,False,Recursive TS,MPS (Quimb),erdos_renyi
...,...,...,...,...,...,...,...,...
1707,001_100nodes_random6regular.json,TQA_MPSAer_opt.json,10,MPS,True,TQA*,MPS (Aer),random_regular
1708,001_100nodes_random9regular.json,TQA_MPSAer_opt.json,10,MPS,True,TQA*,MPS (Aer),random_regular
1709,000_100nodes_1swap_layers.json,FA_MPSAer_opt.json,10,MPS,True,Fixed Angle*,MPS (Aer),line_to_full
1710,002_100nodes_1swap_layers.json,FA_MPSAer_opt.json,10,MPS,True,Fixed Angle*,MPS (Aer),line_to_full


In [32]:
# Save missing instances to CSV.
df_missing[["instance", "method", "depth"]].to_csv(
    f"summary_missing_{table.problem_class}.csv"
)

### Pivot Table of Missing Instances


In [33]:
pivot_missing = df_missing.pivot_table(
    values="instance",
    index=["evaluation_label", "method_label"],
    columns=["graph_type"],
    aggfunc="count",
)
pivot_missing = pivot_missing.reindex(index=row_index)
display(f"Missing Runs for problem_class={table.problem_class} at P={missing_depth}")
display(
    pivot_missing.style.format(precision=0, na_rep=" ")
    .background_gradient("RdYlGn_r")  # pyright: ignore[reportAttributeAccessIssue]
    .highlight_null("transparent")
)

'Missing Runs for problem_class=MC at P=10'